In [456]:
from pathlib import Path
import pandas as pd
import numpy as np

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50_split.csv"
)

df = pd.read_csv(data_file)

print("Dataset loaded!")
print("Shape:", df.shape)

print()
print(df["split"].value_counts())

Dataset loaded!
Shape: (12515, 24)

split
train         8789
test          1866
validation    1860
Name: count, dtype: int64


In [457]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

target = "future_spo2_5min"

In [458]:
train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

In [459]:
train_model_df = train_df.dropna(
    subset=[target]
).copy()

val_model_df = val_df.dropna(
    subset=[target]
).copy()

In [460]:
print(
    "Training rows:",
    len(train_model_df)
)

print(
    "Validation rows:",
    len(val_model_df)
)

Training rows: 8310
Validation rows: 1762


In [461]:
X_train = train_model_df[spo2_features]

y_train = train_model_df[target]

X_val = val_model_df[spo2_features]

y_val = val_model_df[target]

In [462]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print()

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (8310, 8)
y_train: (8310,)

X_val: (1762, 8)
y_val: (1762,)


In [463]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [464]:
linear_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            LinearRegression()
        )
    ]
)

In [465]:
linear_model.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['spo2','spo2_lag_1','spo2_lag_3',...,'spo2_change_3min', 'spo2_rolling_mean_5','spo2_rolling_std_5']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for 

In [466]:
y_val_pred = linear_model.predict(
    X_val
)

In [467]:
comparison = pd.DataFrame({
    "Actual SpO2": y_val.values,
    "Predicted SpO2": y_val_pred
})

comparison.head(10)

,Actual SpO2,Predicted SpO2
0,100.0,99.787643
1,99.0,98.348666
2,100.0,95.514007
3,99.0,88.609789
4,100.0,99.212598
5,99.0,98.921997
6,99.0,94.587787
7,100.0,97.354648
8,100.0,98.960951
9,99.5,99.850408


In [468]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [469]:
linear_mae = mean_absolute_error(
    y_val,
    y_val_pred
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_pred
    )
)

linear_r2 = r2_score(
    y_val,
    y_val_pred
)

In [470]:
print("=== LINEAR REGRESSION ===")
print()

print(
    f"MAE:  {linear_mae:.3f}"
)

print(
    f"RMSE: {linear_rmse:.3f}"
)

print(
    f"R²:   {linear_r2:.3f}"
)

=== LINEAR REGRESSION ===

MAE:  0.487
RMSE: 1.815
R²:   0.089


In [471]:
baseline_file = (
    project_root
    / "data"
    / "processed"
    / "baseline_results.csv"
)

baseline_results = pd.read_csv(
    baseline_file
)

baseline_results

,model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086


In [472]:
linear_results = pd.DataFrame({
    "model": [
        "Linear Regression - SpO2 History"
    ],

    "MAE": [
        linear_mae
    ],

    "RMSE": [
        linear_rmse
    ],

    "R2": [
        linear_r2
    ]
})

In [473]:
model_results = pd.concat(
    [
        baseline_results,
        linear_results
    ],
    ignore_index=True
)

model_results

,model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
1,Linear Regression - SpO2 History,0.487236,1.815459,0.089000


In [474]:
baseline_mae = (
    baseline_results
    .loc[0, "MAE"]
)

mae_improvement = (
    (
        baseline_mae - linear_mae
    )
    /
    baseline_mae
) * 100

print(
    f"MAE improvement over baseline: "
    f"{mae_improvement:.2f}%"
)

MAE improvement over baseline: -82.57%


In [475]:
results_file = (
    project_root
    / "data"
    / "processed"
    / "model_results.csv"
)

model_results.to_csv(
    results_file,
    index=False
)

print("Model results saved!")

Model results saved!


In [476]:
X_val_common = common_val_df[spo2_features]

y_val_common = common_val_df[
    "future_spo2_5min"
]

In [477]:
print("X validation shape:", X_val_common.shape)
print("y validation shape:", y_val_common.shape)

X validation shape: (1748, 8)
y validation shape: (1748,)


In [478]:
common_val_df = val_df.dropna(
    subset=[
        "spo2",
        "future_spo2_5min"
    ]
).copy()

print("Common validation rows:", len(common_val_df))

Common validation rows: 1748


In [479]:
y_linear_common = linear_model.predict(
    X_val_common
)

In [480]:
y_linear_common = linear_model.predict(
    X_val_common
)

In [481]:
linear_mae_common = mean_absolute_error(
    y_val_common,
    y_linear_common
)

linear_rmse_common = np.sqrt(
    mean_squared_error(
        y_val_common,
        y_linear_common
    )
)

linear_r2_common = r2_score(
    y_val_common,
    y_linear_common
)

print("=== LINEAR REGRESSION - COMMON VALIDATION SET ===")
print()

print(f"MAE:  {linear_mae_common:.3f}")
print(f"RMSE: {linear_rmse_common:.3f}")
print(f"R²:   {linear_r2_common:.3f}")

=== LINEAR REGRESSION - COMMON VALIDATION SET ===

MAE:  0.420
RMSE: 1.363
R²:   0.119


In [482]:
y_baseline_common = common_val_df["spo2"]

baseline_mae_common = mean_absolute_error(
    y_val_common,
    y_baseline_common
)

baseline_rmse_common = np.sqrt(
    mean_squared_error(
        y_val_common,
        y_baseline_common
    )
)

baseline_r2_common = r2_score(
    y_val_common,
    y_baseline_common
)

print("=== PERSISTENCE BASELINE - COMMON VALIDATION SET ===")
print()

print(f"MAE:  {baseline_mae_common:.3f}")
print(f"RMSE: {baseline_rmse_common:.3f}")
print(f"R²:   {baseline_r2_common:.3f}")

=== PERSISTENCE BASELINE - COMMON VALIDATION SET ===

MAE:  0.267
RMSE: 1.657
R²:   -0.303


In [483]:
fair_results = pd.DataFrame({
    "Model": [
        "Persistence Baseline",
        "Linear Regression - SpO2 History"
    ],
    "MAE": [
        baseline_mae_common,
        linear_mae_common
    ],
    "RMSE": [
        baseline_rmse_common,
        linear_rmse_common
    ],
    "R2": [
        baseline_r2_common,
        linear_r2_common
    ]
})

fair_results

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610


In [484]:
mae_improvement_common = (
    (
        baseline_mae_common
        - linear_mae_common
    )
    / baseline_mae_common
) * 100

print(
    f"Linear Regression MAE improvement "
    f"over baseline: "
    f"{mae_improvement_common:.2f}%"
)

Linear Regression MAE improvement over baseline: -57.55%


In [485]:
results_file = (
    project_root
    / "data"
    / "processed"
    / "model_results.csv"
)

fair_results.to_csv(
    results_file,
    index=False
)

print("Corrected model results saved!")

Corrected model results saved!


In [486]:
fair_results

,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
